# Assignment 5: Implement a Single-Layer Perceptron
**EMEA AI Cohort 12 — Hands-on Assignment**

## Objective
Implement and evaluate a single-layer perceptron for binary classification, then visualize its linear decision boundary.

**Dataset:** Scikit-learn Breast Cancer Wisconsin dataset (built in; no download needed).  
**Features:** Mean radius and mean texture.  
**Target:** 0 = malignant, 1 = benign.

> Results are calculated when you run the notebook. This is an educational exercise, not a medical diagnostic tool.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load and inspect the dataset

In [ ]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()

feature_names = ["mean radius", "mean texture"]
X = df[feature_names].copy()
y = df["target"].astype(int)  # 0 = malignant, 1 = benign

print("Dataset shape:", df.shape)
print("Class counts (0 = malignant, 1 = benign):")
display(y.value_counts().sort_index().rename_axis("target").to_frame("count"))
display(X.head())

## 3. Split and scale the data

Use a stratified 80/20 train/test split. Fit the scaler only on training data to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 4. Train the single-layer perceptron

A perceptron learns a linear decision boundary by adjusting weights during training.

In [ ]:
perceptron = Perceptron(
    max_iter=1000, eta0=0.01, random_state=RANDOM_STATE, tol=1e-3
)
perceptron.fit(X_train_scaled, y_train)

print("Training completed.")
print("Learned coefficients:", perceptron.coef_)
print("Learned intercept:", perceptron.intercept_)

## 5. Evaluate the model

Metrics are calculated on the held-out test set. The positive class is 1 (benign).

In [ ]:
y_pred = perceptron.predict(X_test_scaled)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (benign)", "Recall (benign)", "F1-score (benign)"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    ]
})
display(metrics.style.format({"Score": "{:.4f}"}))

print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Malignant", "Benign"], zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=[0, 1],
    display_labels=["Malignant", "Benign"], cmap="Blues", values_format="d"
)
plt.title("Perceptron Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 6. Visualize the decision boundary

In [ ]:
x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

mesh_points = np.c_[xx.ravel(), yy.ravel()]
Z = perceptron.predict(mesh_points).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5, 0.5, 1.5])
scatter = plt.scatter(
    X_train_scaled[:, 0], X_train_scaled[:, 1],
    c=y_train, edgecolor="k", alpha=0.7
)
plt.xlabel("Mean radius (standardized)")
plt.ylabel("Mean texture (standardized)")
plt.title("Single-Layer Perceptron Decision Regions")
plt.tight_layout()
plt.show()

## 7. Inspect misclassified examples

In [ ]:
test_results = X_test.copy()
test_results["actual"] = y_test
test_results["predicted"] = y_pred
test_results["correct"] = test_results["actual"] == test_results["predicted"]

print("Misclassified test samples:", int((~test_results["correct"]).sum()))
display(test_results.loc[~test_results["correct"]].head(10))

## 8. Conclusion and submission checklist

### Conclusion template
Replace placeholders with the metrics produced by your run:

> I implemented a single-layer perceptron using mean radius and mean texture from the Breast Cancer Wisconsin dataset. I used a stratified 80/20 train/test split and standardized features using training data only. The model achieved **[measured accuracy]** accuracy, **[measured precision]** precision, **[measured recall]** recall, and **[measured F1-score]** F1-score for the benign class. The confusion matrix summarizes classification errors, and the decision-region plot illustrates the linear decision boundary. Since the perceptron is a linear classifier using two features, it may not capture every pattern. This is an educational exercise, not a medical diagnostic system.

### Checklist
- [ ] Run all cells in order.
- [ ] Include the metrics table, confusion matrix, and decision-boundary plot.
- [ ] Replace placeholders with measured results.
- [ ] Save the notebook or export code as `.py` if code-only submission is required.